# NVIDIA LLM에게 직접 질문해 봅시다

교안에서 본 LLM을 오늘은 직접 사용해 봅니다.

이번 실습에서 해 볼 일은 세 가지뿐입니다.

1. 실제 NVIDIA LLM에게 질문을 보내 봅니다.
2. 같은 LLM에게 **지시만 바꾸어** 서로 다른 형태의 답을 받아 봅니다.
3. **우리가 가진 특정 문서**를 물어보면 어떻게 되는지 확인합니다.

---

## API Key를 한 번 입력합니다

앞에서 NVIDIA에서 발급받은 본인의 API Key를
아래 코드 셀의 따옴표 안에 붙여 넣으세요.

```
NVIDIA_API_KEY = "여기에 붙여넣기"
```

해야 할 일은 딱 세 가지입니다.

1. 복사해 둔 NVIDIA API Key를 복사
2. 아래 셀의 지정된 한 줄에 붙여 넣기
3. ▶ 실행

🔐 API Key는 다른 사람에게 공유하지 않습니다.
실습 후 이 Notebook을 다른 사람에게 보내거나 공유할 경우에는,
입력한 Key를 반드시 지운 뒤 공유합니다.

In [ ]:
# 실행 0 — 실습에 필요한 라이브러리 설치 (uv 환경, 최초 1회만 실행하면 됩니다)
import sys
!uv pip install --python "{sys.executable}" openai

In [ ]:
try:
    from openai import OpenAI
except ImportError:
    raise ImportError(
        "LLM 실습에 필요한 라이브러리가 현재 Python 환경에 설치되어 있지 않습니다. "
        "강사가 안내한 uv 환경이 올바르게 선택되어 있는지 확인해 주세요."
    ) from None


NVIDIA_API_KEY = "여기에_본인의_API_Key를_붙여넣으세요"


if NVIDIA_API_KEY == "여기에_본인의_API_Key를_붙여넣으세요":
    raise ValueError(
        "먼저 위의 NVIDIA_API_KEY 한 줄에 본인의 API Key를 붙여 넣어 주세요."
    )


MODEL = "meta/llama-3.1-70b-instruct"

client = OpenAI(
    base_url="https://integrate.api.nvidia.com/v1",
    api_key=NVIDIA_API_KEY,
)


def 질문하기(질문, 자료=""):
    내용 = 질문
    if 자료:
        내용 += "\n\n참고 자료:\n" + 자료

    try:
        response = client.chat.completions.create(
            model=MODEL,
            messages=[
                {
                    "role": "system",
                    "content": (
                        "한국어로 쉽고 간결하게 답하세요. "
                        "현재 대화에 제공되지 않은 특정 문서의 내용을 "
                        "실제로 확인한 것처럼 말하지 마세요. "
                        "정확한 근거가 필요한데 자료가 제공되지 않았다면, "
                        "해당 자료가 필요하다고 안내하세요."
                    ),
                },
                {"role": "user", "content": 내용},
            ],
            temperature=0.2,
            max_tokens=500,
        )
        print(response.choices[0].message.content)
    except Exception:
        print(
            "LLM에 연결하지 못했습니다. "
            "API Key와 인터넷 연결을 확인해 주세요."
        )


print("준비 완료 — 이제 LLM에게 질문할 수 있습니다.")

## 첫 번째 질문

방금 준비한 NVIDIA LLM에게 아주 간단한 질문을 보내 보겠습니다.

실행하기 전에 잠깐 — **어떤 답이 나올 것 같나요?**

In [ ]:
질문하기(
    "LLM이 무엇인지 한 문장으로 설명해 주세요."
)

방금 여러분이 입력한 질문이 NVIDIA에서 실행되는 LLM에 전달되고,
답변이 돌아왔습니다. 여러분은 지금 실제 LLM에게 질문한 것입니다.

---

## 같은 AI에게 다른 일을 시켜 봅시다

아래 셀에는 짧은 교육 안내문이 준비되어 있습니다.

같은 LLM에게 같은 안내문을 주면서,

- 한 번은 **"요약해 주세요"**
- 한 번은 **"표로 정리해 주세요"**

라고 서로 다른 지시를 보내 봅니다.

In [ ]:
교육안내 = """
AI 활용 교육은 8월 20일 오전 10시에 시작합니다.
장소는 3층 교육장이며,
참석자는 노트북을 지참해 주세요.
교육자료는 당일 배포합니다.
"""

질문하기("다음 내용을 세 줄로 요약해 주세요.", 교육안내)

print()
print("=" * 40)
print()

질문하기("같은 내용을 표로 정리해 주세요.", 교육안내)

## 모델은 같은데, 지시만 바꿨습니다

두 번 모두 **같은 NVIDIA LLM**을 사용했습니다.
달라진 것은 우리가 준 지시뿐입니다.

```
요약해 주세요       →   요약 형태의 답
표로 정리해 주세요   →   표 형태의 답
```

교안에서 본 **"같은 AI 하나, 지시만 바꿉니다"**를
실제로 확인한 것입니다.

---

## 그런데 우리가 가진 특정 문서를 물어보면?

우리 교육 폴더에는 다음 시간에 사용할
**「2. 2026년 세제개편안 상세본」** 문서(PDF·HWPX)가 있습니다.

이번에는 이 문서의 내용을 **함께 주지 않은 채**,
문서에 대해 물어보겠습니다.

실행하기 전에 잠깐 — 이번에는 어떤 답이 나올까요?

In [ ]:
질문하기(
    "우리 교육에서 사용할 "
    "'2. 2026년 세제개편안 상세본'의 "
    "주요 변경 내용을 3가지 알려 주세요."
)

## 왜 정확한 문서 답이라고 가정할 수 없을까요?

지금 이 Notebook의 단순 LLM 호출에는
우리 교육 폴더의 세제개편안 문서나
별도의 검색 기능이 **연결되어 있지 않습니다.**

LLM이 관련된 내용을 학습 과정에서 접했을 수는 있습니다.
하지만 문장이 자연스럽게 읽히더라도,
우리가 가진 **그 문서의 정확한 버전과 내용**을
실제로 확인한 답이라고 가정할 수는 없습니다.

교안에서 본 **"그럴듯하지만 틀릴 수 있습니다"**가 바로 이 이야기입니다.

---

## 왜 다음에 RAG를 배울까요?

```
일반적인 질문
    ↓
LLM에게 바로 질문


같은 자료 + 다른 지시
    ↓
같은 LLM이
다른 형태의 답 생성


우리가 가진 특정 문서 질문
    ↓
현재 Notebook에는
그 문서가 연결되어 있지 않음


문서 내용을 직접 함께 주면?
    ↓
그 내용을 참고할 수 있음


그런데 관련 자료가 여러 개라면?
    ↓
어떤 자료의 어느 부분을
먼저 골라서 줄까?
```

그래서 오늘은 이 질문 하나를 남겨 두고 마칩니다.

> **"질문에 맞는 자료를 먼저 찾아서
> LLM에게 전달하게 할 수는 없을까요?"**

**다음 실습 — RAG**